# 🎯 YOLO26-nano Training — BGMI Killfeed Detection (Backup Model)

**Ye notebook kya karta hai:** YOLO26-nano train karta hai — RF-DETR ka backup. CPU pe fast chalta hai.

## Kaise chalaye (sirf 5 steps)

1. **Kaggle pe account banao** (kaggle.com — free hai)
2. **Dataset upload karo:** Neeche Cell 2 me dataset ka naam dalo
3. **GPU on karo:** Right side me ⚙️ Settings → Accelerator → **GPU T4** select karo
4. **Run dabao:** Top me ▶️ "Run All" dabao
5. **Model download karo:** Last cell se trained model download karo

⏱️ **Time:** 100k images pe ~4-8 ghante (T4 GPU). YOLO RF-DETR se fast train hota hai.

---
## Hyperparameters (research-based)

| Setting | Value | Kyun |
|---|---|---|
| Learning rate | 0.01 (SGD) | YOLOv12 paper ka official value, linear decay to 1e-4 |
| Batch size | 32 | T4 pe nano model aaram se handle karta hai |
| Epochs | 30 | COCO pretrained se fine-tune, early stopping ke saath |
| Image size | 800 | Killfeed text chhota hai — 640 se bada = behtar detection |
| Patience | 10 | 10 epochs me improvement nahi → training rok do (time bachao) |
| Mosaic | 1.0 (last 10 epochs off) | Standard YOLO augmentation |
| STAL | Built-in | YOLO26 ka Small-Target-Aware Label Assignment — chhote objects ke liye |

Sources: YOLOv12 paper (arxiv.org), YOLO26 docs (docs.ultralytics.com), Ultralytics training guide.

In [ ]:
# Cell 1: Packages install karo
!pip install -q ultralytics
print("✅ Ultralytics installed")

In [ ]:
# Cell 2: Dataset load karo
# 👇 APNA DATASET PATH YAHAN DALO
DATASET_PATH = "/kaggle/input/bgmi-killfeed-dataset"  # <-- change karo agar naam alag hai

import os
data_yaml = os.path.join(DATASET_PATH, "data.yaml")
assert os.path.exists(data_yaml), f"data.yaml nahi mila: {data_yaml}"
print(f"✅ Dataset mil gaya")

train_imgs = len(os.listdir(os.path.join(DATASET_PATH, "train", "images")))
val_imgs = len(os.listdir(os.path.join(DATASET_PATH, "valid", "images")))
print(f"📊 Train: {train_imgs:,} images | Val: {val_imgs:,} images")

In [ ]:
# Cell 3: GPU check
import torch
assert torch.cuda.is_available(), "❌ GPU nahi mila! Settings → Accelerator → GPU T4 karo"
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 4: TRAINING SHURU 🚀

from ultralytics import YOLO

# COCO pretrained nano model load karo
model = YOLO("yolo26n.pt")

# Train karo — best hyperparameters
results = model.train(
    data="/kaggle/input/bgmi-killfeed-dataset/data.yaml",
    epochs=30,           # fine-tune ke liye kaafi, early stopping bhi hai
    imgsz=800,           # chhote killfeed text ke liye 640 se behtar
    batch=32,            # T4 pe nano aaram se
    lr0=0.01,            # YOLO paper ka official initial LR (SGD)
    lrf=0.01,            # final LR = 0.01 * 0.01 = 1e-4 (linear decay)
    momentum=0.937,      # YOLO official
    weight_decay=0.0005, # YOLO official
    warmup_epochs=3,     # pehle 3 epochs slow start (stability ke liye)
    patience=10,         # 10 epochs no improvement → stop (time bachao)
    mosaic=1.0,          # standard augmentation
    close_mosaic=10,     # last 10 epochs mosaic band (clean fine-tune)
    device=0,            # GPU use karo
    project="/kaggle/working/yolo26-killfeed",
    name="train",
    exist_ok=True,
    plots=True,          # training graphs banao
)

print("✅ Training complete!")

In [ ]:
# Cell 5: Evaluation — imandaar numbers dekho

from ultralytics import YOLO

model = YOLO("/kaggle/working/yolo26-killfeed/train/weights/best.pt")

# Validation pe test karo
metrics = model.val()
print(f"")
print(f"📊 RESULTS (validation set pe):")
print(f"   mAP@50:    {metrics.box.map50:.3f}")
print(f"   mAP@50-95: {metrics.box.map:.3f}")
print(f"   Precision: {metrics.box.mp:.3f}")
print(f"   Recall:    {metrics.box.mr:.3f}")
print(f"")
print(f"📝 Kya achha hai?")
print(f"   mAP@50 > 0.80  →  Bahut achha ✅")
print(f"   mAP@50 0.60-0.80 →  Kaam chalega, aur data chahiye ⚠️")
print(f"   mAP@50 < 0.60  →  Problem hai, neeche troubleshooting dekho ❌")

In [ ]:
# Cell 6: ONNX me export karo

from ultralytics import YOLO

model = YOLO("/kaggle/working/yolo26-killfeed/train/weights/best.pt")

# ONNX export (AEGIS ke liye)
model.export(format="onnx", imgsz=800)
print("✅ ONNX model ready!")
print("")
print("📥 AB KYA KARNA HAI:")
print("1. Right panel me 'Output' me jao")
print("2. best.onnx download karo")
print("3. AEGIS me VISION_MODEL_PATH me ye path dalo")
print("4. Done! 🎉")

print("")
print("⚠️ LICENSE NOTE: YOLO AGPL-3.0 hai.")
print("   Agar AEGIS bechna hai to RF-DETR wala model use karo (Apache 2.0).")
print("   Ye sirf backup/testing ke liye hai.")